# SAXS NAF Benchmark

Compares NAF against mumott baselines (SH, GK, FBP) on two partial datasets
from the b411 remounting experiment, using the combined (full-coverage) dataset
as ground truth.

**Design**
- Ground truth: mumott SH or GK reconstruction on `combined_dc` (flag: `GT_METHOD`)
- Partial datasets: `b411R_inf_1` and `b411R_remount` independently
- All methods trained on the same 90 % of projections; 10 % held out for
  consistency evaluation
- Metrics: RSM Pearson correlation (bucketed by missing-arc), orientation angular
  error (RA-weighted), RA MAE, fiber symmetry MAE, PSNR/SSIM/NRMSE, held-out NRMSE
- Reconstructions cached as `{name}_{hash8}.npy` + JSON sidecar so the notebook
  can be re-run without recomputing expensive reconstructions


In [1]:
# !pip install pandas
# !pip install pytest

In [2]:
import sys
sys.path.insert(0, '/myhome/smartt')

import copy
import pickle
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mumott.data_handling import DataContainer
from mumott.methods.basis_sets import SphericalHarmonics, GaussianKernels, NearestNeighbor
from mumott.methods.projectors import SAXSProjector
from mumott.methods.residual_calculators import GradientResidualCalculator
from mumott.optimization.loss_functions import SquaredLoss
from mumott.optimization.optimizers import LBFGS
from mumott.optimization.regularizers import Laplacian

from smartt.saxs_naf import saxs_naf_reconstruction
from smartt.saxs_naf.cache import save_recon, load_recon, list_cache
from smartt.saxs_naf.metrics import (
    split_holdout, to_sh_coefficients, compute_ground_truth, compute_metrics, metrics_table
)
from smartt.saxs_fbp.reconstruction import saxs_fbp_reconstruction
from smartt.saxs_naf.eval import plot_rsm_direction, evaluate_models

INFO:Setting the number of threads to 8. If your physical cores are fewer than this number, you may want to use numba.set_num_threads(n), and os.environ["OPENBLAS_NUM_THREADS"] = f"{n}" to set the number of threads to the number of physical cores n.
INFO:Setting numba log level to WARNING.


## Parameters
Edit this cell to change the run configuration.

In [17]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/myhome/data/smartt/shared/b411')
CACHE_DIR  = Path('cache/saxs_naf_benchmark')   # relative to notebooks/
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PATH_B411R    = DATA_DIR / 'dataset_b411R_inf_1_0.220_1.900.h5'
PATH_REMOUNT  = DATA_DIR / 'dataset_b411R_inf1_remount_0.220_1.900.h5'
PATH_GEO_COMBINED = DATA_DIR / 'combined_geometry.h5'
PATH_ABSORPTION   = DATA_DIR / 'sirt_tomogram.h5'
PATH_OTSU_MASK    = DATA_DIR / 'Otsu_mask_arrays.npy'

# ── Benchmark config ───────────────────────────────────────────────────────
GT_METHOD      = 'sh'      # 'sh' or 'gk' — method for ground-truth reconstruction
ELL_MAX        = 8
K_DIRS         = 30        # Fibonacci RSM evaluation directions
HOLDOUT_FRAC   = 0.10      # fraction of projections held out per dataset
HOLDOUT_SEED   = 42

# ── Mumott LBFGS baseline config ───────────────────────────────────────────
# LBFGS converges fast — 20 iterations matches the comparison notebook.
# Do NOT set this to 500+: each LBFGS step on a 141^3 volume is expensive.
MUMOTT_ITERS      = 20
LAPLACIAN_WEIGHT  = 1e-1
SH_ELL_MAX        = 8
# scipy L-BFGS-B Fortran code computes wa_size=(2*maxcor+5)*n with 32-bit ints.
# For n=99.3M (141×111×141×45) maxcor≥9 overflows INT32_MAX → segfault.
MUMOTT_MAXCOR     = 5

# ── NAF config ─────────────────────────────────────────────────────────────
NAF_PARAMS = dict(
    ell_max        = ELL_MAX,
    n_iterations   = 2000,
    lr             = 1e-2,
    batch_size     = 40,
    reg_weight_sh  = 1e-6,
    reg_weight_tv  = 0.,
)

FORCE_RECOMPUTE = False    # set True to ignore all caches and recompute everything

## 1. Load datasets and build combined DC

In [ ]:
# import copy

# def build_combined_dc(path_ds1, path_ds2, path_geo):
#     """Combine both mounts into a single DataContainer for ground-truth reconstruction."""
#     combined = DataContainer(str(path_ds1))
#     for frame in combined.projections:
#         j_pad = 65 - frame.data.shape[0]
#         k_pad = 66 - frame.data.shape[1]
#         frame.diode   = np.pad(frame.diode,   ((j_pad, 0), (k_pad, 0)))
#         frame.data    = np.pad(frame.data,    ((j_pad, 0), (k_pad, 0), (0, 0)))
#         frame.weights = np.pad(frame.weights, ((j_pad, 0), (k_pad, 0), (0, 0)))

#     dc2_copy = DataContainer(str(path_ds2))
#     for frame in dc2_copy.projections:
#         j_pad = 65 - frame.data.shape[0]
#         k_pad = 66 - frame.data.shape[1]
#         frame.diode   = np.pad(frame.diode,   ((j_pad, 0), (k_pad, 0)))
#         frame.data    = np.pad(frame.data,    ((j_pad, 0), (k_pad, 0), (0, 0)))
#         frame.weights = np.pad(frame.weights, ((j_pad, 0), (k_pad, 0), (0, 0)))

#     n = len(dc2_copy.projections)
#     for _ in range(n):
#         frame = dc2_copy.projections[0]
#         del dc2_copy.projections[0]
#         combined.projections.append(frame)

#     combined.geometry.read(str(path_geo))
#     return combined


# DATA_DIR  = Path('/myhome/data/smartt/shared/missing_wedge/full_data_zenodo/data')
# CACHE_DIR = Path('cache/saxs_naf_benchmark_zenodo')   # separate cache from b411

# PATH_B411R        = DATA_DIR / 'data_set_1.h5'        # first mount
# PATH_REMOUNT      = DATA_DIR / 'data_set_2.h5'        # second mount
# PATH_GEO_COMBINED = DATA_DIR / 'full_geometry.geo'    # .geo not .h5
# # PATH_ABSORPTION and PATH_OTSU_MASK: no equivalents — drop them, set otsu_mask = None

# combined_dc = build_combined_dc(PATH_B411R, PATH_REMOUNT, PATH_GEO_COMBINED)

In [4]:
# ── Load b411R (first mount) ───────────────────────────────────────────────
dc_b411r = DataContainer(str(PATH_B411R))
dc_b411r.geometry.full_circle_covered = False   # detector spans ~180°, not 360°

# ── Load remount ───────────────────────────────────────────────────────────
dc_remount = DataContainer(str(PATH_REMOUNT))
dc_remount.geometry.full_circle_covered = False

# ── Build combined DC (both mounts; used for ground truth only) ────────────
combined_dc = DataContainer(str(PATH_B411R))
combined_dc.geometry.full_circle_covered = False

# Pad b411R projections to match combined geometry detector width
for proj in combined_dc.projections:
    proj.diode   = np.pad(proj.diode,   ((0,0),(27,28)),        mode='constant', constant_values=1)
    proj.data    = np.pad(proj.data,    ((0,0),(27,28),(0,0)),   mode='constant', constant_values=0)
    proj.weights = np.pad(proj.weights, ((0,0),(27,28),(0,0)),   mode='constant', constant_values=0)

# Append remount projections (scaled by 0.75 as in original notebook)
dc_rem_copy = copy.deepcopy(dc_remount)
n_rem = len(dc_rem_copy.projections)
for i in range(n_rem):
    dc_rem_copy.projections[0].data *= 0.75
    frame = dc_rem_copy.projections[0]
    del dc_rem_copy.projections[0]
    combined_dc.projections.append(frame)

combined_dc.geometry.read(str(PATH_GEO_COMBINED))

print(f'b411R:    {len(dc_b411r.projections)} projections,  volume {tuple(dc_b411r.geometry.volume_shape)}')
print(f'remount:  {len(dc_remount.projections)} projections, volume {tuple(dc_remount.geometry.volume_shape)}')
print(f'combined: {len(combined_dc.projections)} projections, volume {tuple(combined_dc.geometry.volume_shape)}')

INFO:Rotation matrices were loaded from the input file.
INFO:Sample geometry loaded from file.
INFO:Detector geometry loaded from file.
INFO:Rotation matrices were loaded from the input file.
INFO:Sample geometry loaded from file.
INFO:Detector geometry loaded from file.
INFO:Rotation matrices were loaded from the input file.
INFO:Sample geometry loaded from file.
INFO:Detector geometry loaded from file.
b411R:    266 projections,  volume (np.int64(86), np.int64(111), np.int64(86))
remount:  301 projections, volume (np.int64(141), np.int64(111), np.int64(141))
combined: 567 projections, volume (np.int64(141), np.int64(111), np.int64(141))


In [5]:
# ── Optional: load absorption tomogram and Otsu mask ──────────────────────
otsu_mask = None
if PATH_OTSU_MASK.exists():
    with open(PATH_OTSU_MASK, 'rb') as fid:
        otsu_mask = pickle.load(fid)
    print(f'Otsu mask shape: {otsu_mask.shape}')
else:
    print('No Otsu mask found — auto-Otsu on GT c00 will be used.')

Otsu mask shape: (141, 111, 141)


## 2. Holdout split
Remove 10 % of projections from each partial dataset for held-out evaluation.
The split is random but reproducible via `HOLDOUT_SEED`.

In [6]:
train_b411r, held_b411r = split_holdout(dc_b411r, fraction=HOLDOUT_FRAC, seed=HOLDOUT_SEED)
train_remount, held_remount = split_holdout(dc_remount, fraction=HOLDOUT_FRAC, seed=HOLDOUT_SEED)

print(f'b411R   — train: {len(train_b411r.projections)},  held-out: {len(held_b411r.projections)}')
print(f'remount — train: {len(train_remount.projections)},  held-out: {len(held_remount.projections)}')

b411R   — train: 239,  held-out: 27
remount — train: 271,  held-out: 30


## 3. Ground-truth reconstruction (combined dataset)
Uses `combined_dc` (both mounts → near-full angular coverage).

In [7]:
gt_params = dict(
    gt_method=GT_METHOD,
    ell_max=ELL_MAX,
    n_iterations=MUMOTT_ITERS,
    laplacian_weight=LAPLACIAN_WEIGHT,
    maxcor=5,   # ≤8 required: scipy L-BFGS-B int32 overflow for n≈99M at maxcor≥9
)

coeffs_gt = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'ground_truth', gt_params)
if coeffs_gt is None:
    print('Running ground-truth reconstruction…')
    print('NOTE: one forward pass takes ~2 min; expect ~80 min total. Run via')
    print('  cd notebooks && python ../scripts/compute_benchmark_gt.py')
    print('to avoid kernel timeout, then re-run this cell to load from cache.')
    coeffs_gt = compute_ground_truth(combined_dc, **gt_params)
    save_recon(CACHE_DIR, 'ground_truth', coeffs_gt, gt_params)
    print(f'Saved.  shape={coeffs_gt.shape}')
else:
    print(f'Loaded ground truth from cache  shape={coeffs_gt.shape}')

Loaded ground truth from cache  shape=(141, 111, 141, 45)


## 4. Baselines — b411R (first mount)
All baselines train on `train_b411r` (90 % of b411R projections).

In [8]:
# ── 4a. mumott SphericalHarmonics ──────────────────────────────────────────
sh_params_b411r = dict(method='mumott_sh', ell_max=SH_ELL_MAX, n_iterations=MUMOTT_ITERS, dataset='b411r')

coeffs_sh_b411r = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'mumott_sh_b411r', sh_params_b411r)
if coeffs_sh_b411r is None:
    projector = SAXSProjector(train_b411r.geometry)
    basis_sh = SphericalHarmonics(
        ell_max=SH_ELL_MAX,
        probed_coordinates=train_b411r.geometry.probed_coordinates,
    )
    rc = GradientResidualCalculator(data_container=train_b411r, basis_set=basis_sh, projector=projector)
    loss = SquaredLoss(residual_calculator=rc)
    loss.add_regularizer('laplacian', Laplacian(), regularization_weight=LAPLACIAN_WEIGHT)
    coeffs_sh_b411r = LBFGS(loss, maxiter=MUMOTT_ITERS).optimize()['x'].astype('float32')
    save_recon(CACHE_DIR, 'mumott_sh_b411r', coeffs_sh_b411r, sh_params_b411r)
    print('mumott SH (b411R) done and cached.')
else:
    print(f'Loaded mumott SH (b411R) from cache  shape={coeffs_sh_b411r.shape}')

Loaded mumott SH (b411R) from cache  shape=(86, 111, 86, 45)


In [9]:
# ── 4b. mumott GaussianKernels ─────────────────────────────────────────────
gk_params_b411r = dict(method='mumott_gk', n_iterations=MUMOTT_ITERS, dataset='b411r')

coeffs_gk_b411r = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'mumott_gk_b411r', gk_params_b411r)
if coeffs_gk_b411r is None:
    projector = SAXSProjector(train_b411r.geometry)
    basis_gk = GaussianKernels(probed_coordinates=train_b411r.geometry.probed_coordinates)
    rc = GradientResidualCalculator(data_container=train_b411r, basis_set=basis_gk, projector=projector)
    loss = SquaredLoss(residual_calculator=rc)
    loss.add_regularizer('laplacian', Laplacian(), regularization_weight=LAPLACIAN_WEIGHT)
    result_gk = LBFGS(loss, maxiter=MUMOTT_ITERS).optimize()
    coeffs_gk_b411r = to_sh_coefficients(basis_gk, ell_max=ELL_MAX, coefficients=result_gk['x'])
    save_recon(CACHE_DIR, 'mumott_gk_b411r', coeffs_gk_b411r, gk_params_b411r)
    print('mumott GK (b411R) done and cached.')
else:
    print(f'Loaded mumott GK (b411R) from cache  shape={coeffs_gk_b411r.shape}')

Loaded mumott GK (b411R) from cache  shape=(86, 111, 86, 45)


In [10]:
# ── 4c. FBP ────────────────────────────────────────────────────────────────
fbp_params_b411r = dict(method='fbp', dataset='b411r')

coeffs_fbp_b411r = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'fbp_b411r', fbp_params_b411r)
if coeffs_fbp_b411r is None:
    res_fbp = saxs_fbp_reconstruction(train_b411r)
    coeffs_fbp_b411r = res_fbp['reconstruction'].numpy() if type(res_fbp) == dict and hasattr(res_fbp['reconstruction'], 'numpy') else res_fbp[0].cpu().numpy()
    save_recon(CACHE_DIR, 'fbp_b411r', coeffs_fbp_b411r, fbp_params_b411r)
    print('FBP (b411R) done and cached.')
else:
    print(f'Loaded FBP (b411R) from cache  shape={coeffs_fbp_b411r.shape}')

Loaded FBP (b411R) from cache  shape=(50, 86, 111, 86)


In [18]:
# ── 4d. NAF ────────────────────────────────────────────────────────────────
naf_params_b411r = dict(**NAF_PARAMS, dataset='b411r')

coeffs_naf_b411r = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'naf_b411r', naf_params_b411r)
if coeffs_naf_b411r is None:
    res_naf = saxs_naf_reconstruction(train_b411r, **NAF_PARAMS)
    coeffs_naf_b411r = res_naf['reconstruction'].numpy()
    save_recon(CACHE_DIR, 'naf_b411r', coeffs_naf_b411r, naf_params_b411r)
    print('NAF (b411R) done and cached.')
else:
    print(f'Loaded NAF (b411R) from cache  shape={coeffs_naf_b411r.shape}')

Loaded NAF (b411R) from cache  shape=(86, 111, 86, 45)


## 5. Baselines — remount dataset
Same pipeline on `train_remount`.

In [19]:
# ── 5a. mumott SphericalHarmonics ─────────────────────────────────────────
sh_params_rem = dict(method='mumott_sh', ell_max=SH_ELL_MAX, n_iterations=MUMOTT_ITERS,
                     dataset='remount', maxcor=MUMOTT_MAXCOR)
coeffs_sh_rem = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'mumott_sh_remount', sh_params_rem)
if coeffs_sh_rem is None:
    projector = SAXSProjector(train_remount.geometry)
    basis_sh_rem = SphericalHarmonics(
        ell_max=SH_ELL_MAX,
        probed_coordinates=train_remount.geometry.probed_coordinates,
    )
    rc = GradientResidualCalculator(data_container=train_remount, basis_set=basis_sh_rem, projector=projector)
    loss = SquaredLoss(residual_calculator=rc)
    loss.add_regularizer('laplacian', Laplacian(), regularization_weight=LAPLACIAN_WEIGHT)
    coeffs_sh_rem = LBFGS(loss, maxiter=MUMOTT_ITERS, maxcor=MUMOTT_MAXCOR).optimize()['x'].astype('float32')
    save_recon(CACHE_DIR, 'mumott_sh_remount', coeffs_sh_rem, sh_params_rem)
    print('mumott SH (remount) done and cached.')
else:
    print(f'Loaded mumott SH (remount) from cache  shape={coeffs_sh_rem.shape}')

 60%|██████    | 12/20 [30:27<20:18, 152.27s/it] 
mumott SH (remount) done and cached.


In [20]:
# ── 5b. mumott GaussianKernels ─────────────────────────────────────────────
gk_params_rem = dict(method='mumott_gk', n_iterations=MUMOTT_ITERS,
                     dataset='remount', maxcor=MUMOTT_MAXCOR)
coeffs_gk_rem = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'mumott_gk_remount', gk_params_rem)
if coeffs_gk_rem is None:
    projector = SAXSProjector(train_remount.geometry)
    basis_gk_rem = GaussianKernels(probed_coordinates=train_remount.geometry.probed_coordinates)
    rc = GradientResidualCalculator(data_container=train_remount, basis_set=basis_gk_rem, projector=projector)
    loss = SquaredLoss(residual_calculator=rc)
    loss.add_regularizer('laplacian', Laplacian(), regularization_weight=LAPLACIAN_WEIGHT)
    result_gk_rem = LBFGS(loss, maxiter=MUMOTT_ITERS, maxcor=MUMOTT_MAXCOR).optimize()
    coeffs_gk_rem = to_sh_coefficients(basis_gk_rem, ell_max=ELL_MAX, coefficients=result_gk_rem['x'])
    save_recon(CACHE_DIR, 'mumott_gk_remount', coeffs_gk_rem, gk_params_rem)
    print('mumott GK (remount) done and cached.')
else:
    print(f'Loaded mumott GK (remount) from cache  shape={coeffs_gk_rem.shape}')

 50%|█████     | 10/20 [28:40<28:40, 172.03s/it] 
mumott GK (remount) done and cached.


In [26]:
# ── 5c. FBP ───────────────────────────────────────────────────────────────
fbp_params_rem = dict(method='fbp', dataset='remount')
coeffs_fbp_rem = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'fbp_remount', fbp_params_rem)
if coeffs_fbp_rem is None:
    res_fbp_rem = saxs_fbp_reconstruction(train_remount)
    coeffs_fbp_rem = res_fbp_rem[0].cpu().numpy()
    save_recon(CACHE_DIR, 'fbp_remount', coeffs_fbp_rem, fbp_params_rem)
    print('FBP (remount) done and cached.')
else:
    print(f'Loaded FBP (remount) from cache  shape={coeffs_fbp_rem.shape}')

INFO:Building projection matrix on GPU: K=50, n_samples=64, method=voronoi
INFO:Moving data to cuda...
FBP (remount) done and cached.


In [ ]:
# ── 5d. NAF ───────────────────────────────────────────────────────────────
naf_params_rem = dict(**NAF_PARAMS, dataset='remount')
coeffs_naf_rem = None if FORCE_RECOMPUTE else load_recon(CACHE_DIR, 'naf_remount', naf_params_rem)
if coeffs_naf_rem is None:
    res_naf_rem = saxs_naf_reconstruction(train_remount, **NAF_PARAMS)
    coeffs_naf_rem = res_naf_rem['reconstruction'].numpy()
    save_recon(CACHE_DIR, 'naf_remount', coeffs_naf_rem, naf_params_rem)
    print('NAF (remount) done and cached.')
else:
    print(f'Loaded NAF (remount) from cache  shape={coeffs_naf_rem.shape}')

Calibrated c00 init (mean target/pred = 3.033e+04/1.209e+02).
SAXS-NAF: vol=(np.int64(141), np.int64(111), np.int64(141)) C=45 proj=271 chunks=7 | levels=8 resolutions=[8(dense), 12(dense), 18(dense), 27(dense), 41(dense), 62(dense), 93(dense), 141(dense)]


 25%|██▌       | 502/2000 [07:26<21:44,  1.15it/s, data=3.494e+08, loss=3.513e+08, lr=8.73e-03, sh=1.910e+06]

## 6. Metrics
Compute all quantitative metrics for both datasets.

In [ ]:
metric_kw = dict(
    ground_truth=coeffs_gt,
    ell_max=ELL_MAX,
    K=K_DIRS,
    mask=otsu_mask,   # None → auto-Otsu on GT c00
    half_space='y',
    compute_orientation=True,
)

print('Computing metrics for b411R…')
results_b411r = compute_metrics(
    reconstructions={
        'mumott_sh': coeffs_sh_b411r,
        'mumott_gk': coeffs_gk_b411r,
        'fbp':       coeffs_fbp_b411r,
        'naf':       coeffs_naf_b411r,
    },
    dc=train_b411r,
    held_out_dc=held_b411r,
    **metric_kw,
)

print('Computing metrics for remount…')
results_remount = compute_metrics(
    reconstructions={
        'mumott_sh': coeffs_sh_rem,
        'mumott_gk': coeffs_gk_rem,
        'fbp':       coeffs_fbp_rem,
        'naf':       coeffs_naf_rem,
    },
    dc=train_remount,
    held_out_dc=held_remount,
    **metric_kw,
)
print('Done.')

## 7. Results tables

In [ ]:
print('=== b411R (first mount) ===')
df_b411r = metrics_table(results_b411r)
display(df_b411r.round(4))

In [ ]:
print('=== remount ===')
df_remount = metrics_table(results_remount)
display(df_remount.round(4))

In [ ]:
# RSM correlation bucketed by missing-arc severity
for dataset_label, results in [('b411R', results_b411r), ('remount', results_remount)]:
    print(f'\n=== RSM correlation by missing-arc bucket — {dataset_label} ===')
    rows = {}
    for method, m in results.items():
        if 'rsm_corr_by_arc' in m:
            rows[method] = m['rsm_corr_by_arc']
    if rows:
        display(pd.DataFrame(rows).T.round(4))

## 8. Qualitative visualisations
### 8a. c00 (mean intensity) slices

In [ ]:
def plot_c00_comparison(coeffs_dict, gt, title, slice_axis='z'):
    """Orthogonal c00 slice comparison across methods."""
    methods = list(coeffs_dict.keys())
    n = len(methods) + 1
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    all_coeffs = [gt] + [coeffs_dict[m] for m in methods]
    labels = ['ground_truth'] + methods
    gt_c00 = gt[..., 0]
    lo, hi = np.percentile(gt_c00, [2, 98])
    mid = gt_c00.shape[{'x':0,'y':1,'z':2}[slice_axis]] // 2
    for ax, c, label in zip(axes, all_coeffs, labels):
        c00 = c[..., 0]
        sl = c00[mid] if slice_axis=='x' else (c00[:,mid] if slice_axis=='y' else c00[:,:,mid])
        im = ax.imshow(sl.T, cmap='inferno', vmin=lo, vmax=hi, origin='lower')
        ax.set_title(label, fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(title)
    fig.tight_layout()
    return fig

plot_c00_comparison(
    {'mumott_sh': coeffs_sh_b411r, 'mumott_gk': coeffs_gk_b411r,
     'fbp': coeffs_fbp_b411r, 'naf': coeffs_naf_b411r},
    coeffs_gt, title='c00 (mean intensity) — b411R'
)
plt.show()

### 8b. Per-voxel RSM correlation maps

In [ ]:
def plot_corr_maps(results, title):
    methods = list(results.keys())
    fig, axes = plt.subplots(1, len(methods), figsize=(4*len(methods), 4))
    for ax, method in zip(axes if len(methods)>1 else [axes], methods):
        cmap = results[method]['rsm_corr_map']
        mid = cmap.shape[2] // 2
        im = ax.imshow(cmap[:, :, mid].T, cmap='RdYlGn', vmin=-1, vmax=1, origin='lower')
        ax.set_title(f'{method}\nr={results[method]["rsm_corr_mean"]:.3f}', fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f'RSM Pearson r — {title}')
    fig.tight_layout()
    return fig

plot_corr_maps(results_b411r, 'b411R')
plt.show()
plot_corr_maps(results_remount, 'remount')
plt.show()

### 8c. Orientation error maps

In [ ]:
def plot_orientation_maps(results, title):
    methods = [m for m in results if 'orientation_error_map' in results[m]]
    if not methods:
        print('No orientation maps available (compute_orientation=False?)')
        return
    fig, axes = plt.subplots(1, len(methods), figsize=(4*len(methods), 4))
    for ax, method in zip(axes if len(methods)>1 else [axes], methods):
        err = results[method]['orientation_error_map']
        mid = err.shape[2] // 2
        im = ax.imshow(err[:, :, mid].T, cmap='hot_r', vmin=0, vmax=45, origin='lower')
        ax.set_title(f'{method}\nmean={results[method]["orientation_error_mean"]:.1f}°', fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, label='degrees')
    fig.suptitle(f'Orientation error — {title}')
    fig.tight_layout()
    return fig

plot_orientation_maps(results_b411r, 'b411R')
plt.show()

### 8d. Per-direction RSM volumes (qualitative)
Use `plot_rsm_direction` from `smartt.saxs_naf.eval` to compare a specific
RSM direction across methods.

In [ ]:
# Pick a direction in the missing-wedge region to showcase recovery
K_SHOW = 5   # direction index — adjust after inspecting missing_arcs_deg

cmp_b411r = evaluate_models(
    {
        'GT':         coeffs_gt,
        'mumott_sh':  coeffs_sh_b411r,
        'mumott_gk':  coeffs_gk_b411r,
        'naf':        coeffs_naf_b411r,
    },
    ell_max=ELL_MAX, K=K_DIRS, half_space='y',
    alpha_deg=45,   # b411 tilt angle
)

# Print missing-arc lengths so you can pick a high-wedge direction for K_SHOW
if 'missing_arcs_deg' in cmp_b411r:
    for k, arc in enumerate(cmp_b411r['missing_arcs_deg']):
        print(f'k={k:2d}  arc={arc:.1f}°')

In [ ]:
fig = plot_rsm_direction(cmp_b411r, k=K_SHOW, axis='z')
plt.show()

### 8e. Cache inventory

In [ ]:
for entry in list_cache(CACHE_DIR):
    print(f"{entry['name']:30s}  hash={entry['hash']}  shape={entry.get('shape')}")